## Stage 03b — MML Instrument Alignment

Aligns MML platform instruments using two corrections applied together:

1. **Tube delay** (Sections E1–E3): gas analyzers sample through a tube inlet.
   The H2O spike (e.g., exhaled breath near sensors) arrives at the Anem instantly
   but is delayed at the gas inlet by the tube travel time.
   Reference = Anem wind components (u/v/w); test = gas H2O_ppm (+ other species).
   Correction shifts gas timestamps *earlier* to remove the tube delay.

2. **GPS clock correction** (Section F): the LANL Toughbook clock drifted over the
   campaign (fast by ~0.7 s in January, ~13 s by March 8).
   GPS satellite UTC is the ground truth.  `gps_corrections[date_tag]` = median
   `(toughbook_epoch − GPS_UTC_s)` per date; positive = toughbook fast.

**Total lag applied to gas:** `tube_lag − gps_corr`
**Total lag applied to Anem / GPS:** `−gps_corr`

| Section | Instrument | Reference | Method |
|---|---|---|---|
| E1 | LANL_aerisultra321 (MML dates) | Anem u/v/w | manual H2O spike, normalize=True |
| E2 | LANL_aerispico017  (MML dates) | Anem u/v/w | manual H2O spike, normalize=True |
| E3 | UOU_LGR (Mar 10) | Anem u/v/w | manual H2O spike, normalize=True |
| F  | LANL_GPS | GPS satellite UTC | auto (median offset per date) |

**Outputs:** `lag_offsets_mml.json`, `apply_manifest_mml.json`,
aligned Parquet in `03_instrument_aligned/`.

In [ ]:
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

sys.path.insert(0, str(Path().resolve().parent))
from paths import STAGE_02_DIR, STAGE_03_DIR, REPO_ROOT
from config import MML_DATE_TAGS

display(HTML('<style>.plotly-graph-div { width: 100% !important; }</style>'))
print('Imports OK')

In [ ]:
MAX_LAG_S      = 600
MAX_GPS_CORR_S = 60   # files with |offset| > 60s treated as no-lock

ULTRA321_DIR = STAGE_02_DIR / 'LANL_aerisultra321' / 'Raw'
PICO017_DIR  = STAGE_02_DIR / 'LANL_aerispico017'  / 'Raw'
LGR_DIR      = STAGE_02_DIR / 'UOU_LGR'
ANEM_DIR     = STAGE_02_DIR / 'LANL_Anem'
GPS_DIR      = STAGE_02_DIR / 'LANL_GPS'

print(f'MML_DATE_TAGS ({len(MML_DATE_TAGS)} dates): {sorted(MML_DATE_TAGS)}')
print('Config OK')

In [ ]:
from src.align import resample_series, cross_correlate

def load_parquet_col(path, col):
    return pd.read_parquet(path, columns=[col])[col].dropna()

def load_anem_ref():
    files = sorted(ANEM_DIR.glob('*.parquet'))
    cols  = ['u_ms', 'v_ms', 'w_ms']
    parts = {c: [] for c in cols}
    for f in files:
        df = pd.read_parquet(f, columns=cols)
        for c in cols:
            parts[c].append(df[c].dropna())
    result = {}
    for c in cols:
        s = pd.concat(parts[c]).sort_index()
        s = s[~s.index.duplicated(keep='first')]
        result[c] = resample_series(s)
    return result

WYO_ONLY_STEMS = {'Ultra100460'}

def is_mml(path):
    stem   = Path(path).stem
    prefix = stem.split('_')[0]
    if prefix in WYO_ONLY_STEMS:
        return False
    date_tag = stem.split('_')[1] if stem.count('_') >= 1 else ''
    return date_tag in MML_DATE_TAGS

def _date_tag(path):
    parts = Path(path).stem.split('_')
    return parts[1] if len(parts) >= 2 else ''

def _git_info():
    try:
        h = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], cwd=str(REPO_ROOT), text=True
        ).strip()
        dirty = subprocess.call(['git', 'diff', '--quiet'], cwd=str(REPO_ROOT)) != 0
        return h, dirty
    except Exception:
        return 'unknown', False

def save_lag_offsets_mml():
    STAGE_03_DIR.mkdir(parents=True, exist_ok=True)
    g = globals()
    def _lags(conf, rej):
        return {k: v for k, v in conf.items() if k not in rej}
    git_hash, git_dirty = _git_info()
    manifest = {
        'stage':    '03b_align_mml',
        'run_utc':  datetime.now(timezone.utc).isoformat(),
        'git_hash': git_hash,
        'git_dirty': git_dirty,
        'tube_lags': {
            'LANL_aerisultra321': _lags(g.get('u321_mml_confirmed', {}), g.get('u321_mml_rejected', set())),
            'LANL_aerispico017':  _lags(g.get('pico_mml_confirmed', {}), g.get('pico_mml_rejected', set())),
            'UOU_LGR':            _lags(g.get('lgr_confirmed',      {}), g.get('lgr_rejected',      set())),
        },
        'rejected': {
            'LANL_aerisultra321': sorted(g.get('u321_mml_rejected', set())),
            'LANL_aerispico017':  sorted(g.get('pico_mml_rejected', set())),
            'UOU_LGR':            sorted(g.get('lgr_rejected',      set())),
        },
        'gps_corrections': g.get('gps_corrections', {}),
    }
    with open(STAGE_03_DIR / 'lag_offsets_mml.json', 'w') as fh:
        json.dump(manifest, fh, indent=2)

print('Helpers loaded.')

In [ ]:
def make_review_widget(ref_data, test_files, test_name, suggestions, confirmed, rejected,
                       ref_name='Ref', save_fn=None, test_cols=None, normalize=False):
    """
    Interactive review widget.

    ref_data : pd.Series | dict[str, pd.Series]
        If dict, each key becomes a separate gray reference trace.
    test_cols : list[str] | None
        Columns to load from each test file.  Default: ['CH4_ppm'].
    normalize : bool
        Z-score each trace for cross-unit overlay.
    """
    if not test_files:
        print(f'{test_name}: no files to review')
        return

    state      = {'idx': 0}
    _ref_dict  = ref_data if isinstance(ref_data, dict) else {ref_name: ref_data}
    _n_ref     = len(_ref_dict)
    _test_cols = test_cols or ['CH4_ppm']
    _n_test    = len(_test_cols)

    REF_COLORS  = ['#888888', '#AAAAAA', '#666666', '#BBBBBB']
    TEST_COLORS = ['#E67E22', '#2980B9', '#27AE60', '#8E44AD']

    fig = go.FigureWidget(layout=go.Layout(
        autosize=True, height=400,
        margin=dict(l=55, r=15, t=62, b=40),
        yaxis=dict(title='z-score' if normalize else _test_cols[0]),
        legend=dict(x=1, y=1, xanchor='right', font=dict(size=10)),
        hovermode='x unified',
    ))
    for i, rk in enumerate(_ref_dict.keys()):
        fig.add_scatter(
            name=rk,
            line=dict(color=REF_COLORS[i % len(REF_COLORS)], width=1.5, dash='dot'),
            opacity=0.85,
        )
    for i, col in enumerate(_test_cols):
        fig.add_scatter(
            name=f'{test_name} -- {col}',
            line=dict(color=TEST_COLORS[i % len(TEST_COLORS)], width=2.5),
        )

    lag_slider = widgets.FloatSlider(
        value=0.0, min=-MAX_LAG_S, max=MAX_LAG_S, step=0.1,
        description='Lag (s):', continuous_update=True, readout_format='.1f',
        layout=widgets.Layout(width='100%'),
        style={'description_width': '60px'},
    )

    btn_prev   = widgets.Button(description='<- Prev',         layout=widgets.Layout(width='85px'))
    btn_next   = widgets.Button(description='Next ->',         layout=widgets.Layout(width='85px'))
    btn_commit = widgets.Button(description='Commit & Next',   button_style='success', layout=widgets.Layout(width='140px'))
    btn_bad    = widgets.Button(description='Mark Bad & Next', button_style='danger',  layout=widgets.Layout(width='150px'))
    log = widgets.Output(layout=widgets.Layout(
        height='90px', overflow_y='auto', border='1px solid #ddd', padding='4px',
    ))

    def _zscore(s):
        mu, sig = s.mean(), s.std()
        return (s - mu) / sig if sig > 0 else s * 0.0

    def _update_fig(idx, lag_s):
        if idx >= len(test_files):
            with fig.batch_update():
                for trace in fig.data:
                    trace.x = []; trace.y = []
            fig.layout.title = dict(
                text=f'{test_name} -- complete  ({len(confirmed)} confirmed, {len(rejected)} rejected)',
                y=0.97, yanchor='top',
            )
            return
        f        = test_files[idx]
        key      = f.stem
        auto_lag = suggestions.get(key, 0.0)
        try:
            test_dict = {
                col: resample_series(load_parquet_col(f, col))
                for col in _test_cols
            }
        except Exception as e:
            fig.layout.title = dict(text=f'ERROR loading {f.name}: {e}')
            return
        first_sig = next(iter(test_dict.values()))
        t0 = first_sig.index[0]  - pd.Timedelta(hours=1)
        t1 = first_sig.index[-1] + pd.Timedelta(hours=1)
        with fig.batch_update():
            for i, (rk, rseries) in enumerate(_ref_dict.items()):
                ref_win  = rseries[t0:t1]
                ref_plot = _zscore(ref_win) if normalize else ref_win
                fig.data[i].x = ref_plot.index
                fig.data[i].y = ref_plot.values
            for i, (col, sig) in enumerate(test_dict.items()):
                shifted_idx = sig.index + pd.Timedelta(seconds=lag_s)
                plot_sig    = _zscore(sig) if normalize else sig
                fig.data[_n_ref + i].x = shifted_idx
                fig.data[_n_ref + i].y = plot_sig.values
                fig.data[_n_ref + i].name = f'{col} (lag={lag_s:+.1f}s)'
        status    = f'  v {confirmed[key]:+.1f}s' if key in confirmed else ('  x bad' if key in rejected else '')
        first_ref = next(iter(_ref_dict.values()))
        n_ref_pts = int(first_ref[t0:t1].notna().sum())
        subtitle  = (f'{len(first_sig):,} rows  {first_sig.index[0].strftime("%H:%M")}'
                     f'-{first_sig.index[-1].strftime("%H:%M")} UTC  ref: {n_ref_pts:,} pts')
        fig.layout.title = dict(
            text=(f'[{idx+1}/{len(test_files)}]  {f.name}'
                  f'  auto={auto_lag:+.1f}s{status}<br><sup>{subtitle}</sup>'),
            y=0.97, yanchor='top',
        )

    def go_to(idx):
        if 0 <= idx < len(test_files):
            lag_slider.value = suggestions.get(test_files[idx].stem, 0.0)
        _update_fig(idx, lag_slider.value)

    lag_slider.observe(lambda change: _update_fig(state['idx'], change['new']), names='value')

    def on_prev(_):
        state['idx'] = max(0, state['idx'] - 1); go_to(state['idx'])
    def on_next(_):
        state['idx'] += 1; go_to(state['idx'])
    def on_commit(_):
        if state['idx'] >= len(test_files): return
        key = test_files[state['idx']].stem; lag = round(lag_slider.value, 1)
        confirmed[key] = lag; rejected.discard(key)
        if save_fn: save_fn()
        with log: print(f'COMMITTED  {key}  {lag:+.1f}s')
        state['idx'] += 1; go_to(state['idx'])
    def on_bad(_):
        if state['idx'] >= len(test_files): return
        key = test_files[state['idx']].stem
        rejected.add(key); confirmed.pop(key, None)
        if save_fn: save_fn()
        with log: print(f'REJECTED   {key}')
        state['idx'] += 1; go_to(state['idx'])

    btn_prev.on_click(on_prev); btn_next.on_click(on_next)
    btn_commit.on_click(on_commit); btn_bad.on_click(on_bad)
    btn_row = widgets.HBox([btn_prev, btn_bad, btn_commit, btn_next],
                           layout=widgets.Layout(gap='6px', margin='4px 0'))
    display(widgets.VBox([fig, lag_slider, btn_row, log],
                         layout=widgets.Layout(width='100%')))
    go_to(0)

print('Widget helper loaded.')

In [ ]:
print('Loading LANL_Anem reference...')
anem_ref = load_anem_ref()
print('Anem reference (all MML dates concatenated):')
for k, v in anem_ref.items():
    print(f'  {k}: {len(v):,} samples  {v.index[0]}  ->  {v.index[-1]}')

---
## E1 — Ultra 321 MML dates vs Anem

Tube delay between the gas inlet and the Trisonica anemometer.
Reference traces (gray dotted): Anem u_ms, v_ms, w_ms (z-scored).
Test traces (colored): Ultra 321 H2O_ppm, CH4_ppm, C2H6_ppm, C3H8_ppm (z-scored).

Look for a shared H2O spike event and drag the lag slider until the test traces
align with the Anem reference.  Gas traces shift LEFT for negative lag values
(gas was seeing the event later = gas timestamps were too high).

In [ ]:
u321_all = sorted(ULTRA321_DIR.glob('*.parquet'))
u321_mml = [f for f in u321_all if is_mml(f)]
print(f'Ultra 321 MML-date files: {len(u321_mml)}')
print('Default suggestions: 0s (manual spike alignment)\n')
for i, f in enumerate(u321_mml):
    print(f'[{i:>2}]  {f.name}')
u321_mml_suggestions = {f.stem: 0.0 for f in u321_mml}

In [ ]:
if 'u321_mml_confirmed' not in dir(): u321_mml_confirmed = {}
if 'u321_mml_rejected'  not in dir(): u321_mml_rejected  = set()
make_review_widget(anem_ref, u321_mml, 'Ultra321-MML',
                   u321_mml_suggestions, u321_mml_confirmed, u321_mml_rejected,
                   ref_name='Anem (ref)', save_fn=save_lag_offsets_mml,
                   test_cols=['H2O_ppm', 'CH4_ppm', 'C2H6_ppm', 'C3H8_ppm'],
                   normalize=True)

---
## E2 — Pico 017 MML dates vs Anem

Same spike-alignment procedure as E1.

In [ ]:
pico_all = sorted(PICO017_DIR.glob('*.parquet'))
pico_mml = [f for f in pico_all if is_mml(f)]
print(f'Pico 017 MML-date files: {len(pico_mml)}')
pico_mml_suggestions = {f.stem: 0.0 for f in pico_mml}

In [ ]:
if 'pico_mml_confirmed' not in dir(): pico_mml_confirmed = {}
if 'pico_mml_rejected'  not in dir(): pico_mml_rejected  = set()
make_review_widget(anem_ref, pico_mml, 'Pico017-MML',
                   pico_mml_suggestions, pico_mml_confirmed, pico_mml_rejected,
                   ref_name='Anem (ref)', save_fn=save_lag_offsets_mml,
                   test_cols=['H2O_ppm', 'CH4_ppm', 'C2H6_ppb'],
                   normalize=True)

---
## E3 — LGR vs Anem (Mar 10 Callao Survey)

UOU LGR was present on 2026-03-10 only.  Same spike-alignment procedure.

In [ ]:
lgr_files = sorted(LGR_DIR.glob('*.parquet'))
print(f'UOU_LGR: {len(lgr_files)} file(s)  (Mar 10 MML)')
lgr_suggestions = {f.stem: 0.0 for f in lgr_files}

In [ ]:
if 'lgr_confirmed' not in dir(): lgr_confirmed = {}
if 'lgr_rejected'  not in dir(): lgr_rejected  = set()
make_review_widget(anem_ref, lgr_files, 'LGR',
                   lgr_suggestions, lgr_confirmed, lgr_rejected,
                   ref_name='Anem (ref)', save_fn=save_lag_offsets_mml,
                   test_cols=['H2O_ppm', 'CH4_ppm', 'CO2_ppm'],
                   normalize=True)

---
## F — GPS clock correction

Reads Stage 02 `LANL_GPS` Parquet files which contain both `epoch`
(toughbook Unix timestamp) and `gps_receiver_utc` (true GPS satellite UTC).

`gps_correction[date_tag] = median(epoch − GPS_UTC_s)`

Positive value = toughbook was running fast on that date.
Files with `|offset| > MAX_GPS_CORR_S` (GPS not yet locked) are excluded.

**No widget needed** — GPS UTC is the ground truth; the correction is computed
automatically. Review the table below before proceeding to Apply.

In [ ]:
gps_files = sorted(GPS_DIR.glob('*.parquet'))
print(f'GPS files: {len(gps_files)}\n')

raw_by_date = {}
hdr = f"{'DATE':>8}  {'FILE':<50}  {'N':>6}  {'MEDIAN_OFFSET_S':>16}  STATUS"
print(hdr)
print('-' * len(hdr))
for f in gps_files:
    df = pd.read_parquet(f, columns=['epoch', 'gps_receiver_utc'])
    df = df.dropna(subset=['gps_receiver_utc'])
    if df.empty:
        print(f'          {f.name:<50}  {"---":>6}  {"no GPS fix":>16}')
        continue
    epoch_s   = df['epoch'].astype(float)
    gps_utc_s = df['gps_receiver_utc'].apply(lambda t: t.timestamp())
    offset    = epoch_s - gps_utc_s
    med       = float(offset.median())
    dtag      = _date_tag(f)
    ok        = abs(med) <= MAX_GPS_CORR_S
    status    = 'OK' if ok else f'SKIP (|offset|={abs(med):.0f}s > {MAX_GPS_CORR_S}s)'
    print(f'  {dtag}  {f.name:<50}  {len(offset):>6}  {med:>+16.2f}s  {status}')
    if ok:
        raw_by_date.setdefault(dtag, []).append(med)

gps_corrections = {
    tag: round(float(np.mean(vals)), 3)
    for tag, vals in raw_by_date.items()
}
print(f'\nPer-date GPS corrections (s; positive = toughbook fast):')
for tag, corr in sorted(gps_corrections.items()):
    print(f'  {tag}:  {corr:>+8.3f}s')

---
## Save lag_offsets_mml.json

Saves tube lags (from E sections) and GPS corrections (from F) together.
`lag_offsets_mml.json` is also auto-saved after every Commit/Mark Bad click.

In [ ]:
save_lag_offsets_mml()
lag_path = STAGE_03_DIR / 'lag_offsets_mml.json'
print(f'Saved -> {lag_path}\n')
with open(lag_path) as fh:
    saved = json.load(fh)
print(f"GPS corrections: {saved['gps_corrections']}\n")
for inst, lags in saved['tube_lags'].items():
    rej = saved['rejected'].get(inst, [])
    if lags or rej:
        print(f'{inst}: {len(lags)} tube lags confirmed, {len(rej)} rejected')
        for stem, lag in sorted(lags.items()):
            print(f'  {stem:<55}  {lag:>+6.1f}s')

---
## Apply lags → `03_instrument_aligned/`

**Gas instruments** (Ultra 321, Pico 017, LGR):
  `total_lag = tube_lag − gps_corr`  → `ts_status = 'gps_corrected'`

**Anem and GPS:**
  `total_lag = −gps_corr`  → `ts_status = 'gps_corrected'`

Loads `lag_offsets_mml.json` — safe to re-run.

In [ ]:
import shutil
from src.align import raw_stem, apply_lag_to_parquet

def apply_gas_mml(instrument, subdirs, tube_lags, rejected_stems, apply_spectra=True):
    """Apply total_lag = tube_lag - gps_corr per file to MML gas instrument files."""
    src_inst = STAGE_02_DIR / instrument
    dst_inst = STAGE_03_DIR / instrument
    n_ok = n_bad = n_warn = 0
    for subdir in subdirs:
        if not apply_spectra and subdir in ('Spectra', 'Spectralite'):
            continue
        src_dir = src_inst / subdir if subdir else src_inst
        if not src_dir.exists():
            continue
        for path in sorted(src_dir.glob('*.parquet')):
            if not is_mml(path):
                continue
            rs       = raw_stem(path)
            dst_base = dst_inst / subdir if subdir else dst_inst
            if rs in rejected_stems:
                dst_path = dst_base / 'bad' / path.name
                dst_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, dst_path)
                print(f'  [BAD]  {path.name:<55}  -> bad/')
                n_bad += 1
                continue
            tube_lag = tube_lags.get(rs)
            dtag     = _date_tag(path)
            gps_corr = gps_corrections.get(dtag, 0.0)
            if tube_lag is None:
                print(f'  [WARN no tube lag]  {path.name} — using 0s')
                tube_lag = 0.0; n_warn += 1
            total = tube_lag - gps_corr
            dst_path = dst_base / path.name
            rows = apply_lag_to_parquet(path, total, dst_path, ts_status='gps_corrected')
            print(f'  [OK]  {path.name:<55}  tube={tube_lag:>+5.1f}s  gps={gps_corr:>+6.3f}s  total={total:>+8.3f}s  [{rows:,}]')
            n_ok += 1
    print(f'  -> {instrument}: aligned={n_ok}, bad={n_bad}, warn={n_warn}')
    return {'ok': n_ok, 'bad': n_bad, 'warn': n_warn}

print('MML apply helpers loaded.')

In [ ]:
APPLY_SPECTRA = True

with open(STAGE_03_DIR / 'lag_offsets_mml.json') as fh:
    saved = json.load(fh)

tube_lags_by_inst = saved['tube_lags']
rejected_by_inst  = {k: set(v) for k, v in saved['rejected'].items()}
gps_corrections   = saved['gps_corrections']

apply_stats = {}

# ── Gas instruments: tube delay + GPS correction ─────────────────────────────
GAS_INSTRUMENTS = {
    'LANL_aerisultra321': ['Raw', 'Eng', 'Spectra'],
    'LANL_aerispico017':  ['Raw', 'Eng', 'Spectra'],
    'UOU_LGR':            [''],
}
for inst, subdirs in GAS_INSTRUMENTS.items():
    print(f'\n{"="*60}\n  {inst}\n{"="*60}')
    stats = apply_gas_mml(
        inst, subdirs,
        tube_lags_by_inst.get(inst, {}),
        rejected_by_inst.get(inst, set()),
        apply_spectra=APPLY_SPECTRA,
    )
    apply_stats[inst] = stats

# ── LANL_Anem: GPS correction only ───────────────────────────────────────────
print(f'\n{"="*60}\n  LANL_Anem (GPS correction only)\n{"="*60}')
n_anem = 0
for f in sorted(ANEM_DIR.glob('*.parquet')):
    dtag  = _date_tag(f)
    corr  = gps_corrections.get(dtag, 0.0)
    total = -corr
    rows  = apply_lag_to_parquet(f, total, STAGE_03_DIR / 'LANL_Anem' / f.name,
                                 ts_status='gps_corrected')
    print(f'  [OK]  {f.name:<55}  gps={corr:>+6.3f}s  total={total:>+8.3f}s  [{rows:,}]')
    n_anem += 1
apply_stats['LANL_Anem'] = {'ok': n_anem}

# ── LANL_GPS: GPS correction only ────────────────────────────────────────────
print(f'\n{"="*60}\n  LANL_GPS (GPS correction only)\n{"="*60}')
n_gps = 0
for f in sorted(GPS_DIR.glob('*.parquet')):
    dtag  = _date_tag(f)
    corr  = gps_corrections.get(dtag, 0.0)
    total = -corr
    rows  = apply_lag_to_parquet(f, total, STAGE_03_DIR / 'LANL_GPS' / f.name,
                                 ts_status='gps_corrected')
    print(f'  [OK]  {f.name:<55}  gps={corr:>+6.3f}s  total={total:>+8.3f}s  [{rows:,}]')
    n_gps += 1
apply_stats['LANL_GPS'] = {'ok': n_gps}

apply_manifest = {
    'stage':         '03b_apply_mml',
    'run_utc':       datetime.now(timezone.utc).isoformat(),
    'git_hash':      saved['git_hash'],
    'git_dirty':     saved['git_dirty'],
    'apply_spectra': APPLY_SPECTRA,
    'instruments':   apply_stats,
}
apply_path = STAGE_03_DIR / 'apply_manifest_mml.json'
with open(apply_path, 'w') as fh:
    json.dump(apply_manifest, fh, indent=2)
print(f'\nMML apply complete -> {apply_path}')
print('Run no_coverage pass-through cell below to finish Stage 03b.')

---
## Pass-through: no_coverage → bad_timestamp

Stage 02 `no_coverage/` files have Mountain Time clocks.
Copied to `bad_timestamp/` unchanged.

In [ ]:
NO_COVERAGE_SUBDIRS = {
    'LANL_aerisultra321': ['Raw', 'Eng', 'Spectra'],
    'LANL_aerispico017':  ['Raw', 'Eng', 'Spectra'],
}

passthrough_stats = {}
for inst, subdirs in NO_COVERAGE_SUBDIRS.items():
    n_ok = 0
    for subdir in subdirs:
        if not APPLY_SPECTRA and subdir == 'Spectra':
            continue
        src_dir = STAGE_02_DIR / inst / subdir / 'no_coverage'
        dst_dir = STAGE_03_DIR / inst / subdir / 'bad_timestamp'
        if not src_dir.exists():
            continue
        files = sorted(src_dir.glob('*.parquet'))
        if not files:
            continue
        dst_dir.mkdir(parents=True, exist_ok=True)
        for path in files:
            shutil.copy2(path, dst_dir / path.name)
            print(f'  [PASS]  {inst}/{subdir}/bad_timestamp/{path.name}')
            n_ok += 1
    passthrough_stats[inst] = {'copied': n_ok}
    print(f'  -> {inst}: {n_ok} files -> bad_timestamp/')

apply_path = STAGE_03_DIR / 'apply_manifest_mml.json'
with open(apply_path) as fh:
    m = json.load(fh)
m['passthrough'] = passthrough_stats
with open(apply_path, 'w') as fh:
    json.dump(m, fh, indent=2)
print(f'\nno_coverage pass-through complete.')
print(f'Stage 03b complete -> {STAGE_03_DIR}')